# LSTM Behavioural Model - TrackEasy-3.O

This notebook is a **faithful, descriptive port of [`train_lstm.py`](train_lstm.py)** - the architecture, hyperparameters, dataset, optimiser, loss, metrics, callbacks, and save path are **identical**. The only additions are markdown explanations, EDA cells, and training-curve / evaluation visualisations. Code that affects the trained weights is unchanged.

## Role in the TrackEasy pipeline

The LSTM is the **behavioural specialist** in *Layer 1* of the multi-layered ensemble described in [`fraud_model_explanation.md`](../../../fraud_model_explanation.md):

```
Layer 1 - Specialists      Layer 2 - Rules       Layer 3 - Master Brain
---------------------      ---------------       ----------------------
LSTM   (this notebook)     Superman speed        ANN + XGBoost + RF + IF
GNN    (train_gnn.py)      Velocity spikes       -> fused 0-10 risk score
Autoencoder (train_autoencoder.py)
```

The LSTM consumes the **last 10 events** of a user session (e1...e10) and outputs a fraud probability `lstmProb`. That probability becomes one of the six features fed to the Master-Brain ensemble (alongside `gnnProb`, `autoMSE`, `ruleScore`, `geoSpeed`, `clusterSize`).

## Why LSTM and not a flat classifier?

A 2019-style tabular classifier sees only an aggregated transaction snapshot - it cannot tell the difference between

- *natural browsing*: `login -> browse -> cart -> cart -> checkout`
- *bot smash-and-grab*: `login -> cart -> cart -> cart -> checkout`

The LSTM reads the sequence as a sequence - it learns the temporal pattern, not just the marginal distribution of event codes.


---

## 1. Imports & Configuration

Same imports as [`train_lstm.py`](train_lstm.py). `DATA_PATH` and `MODEL_PATH` are resolved relative to the notebook's working directory (notebooks have no `__file__`, so we use `os.getcwd()`); when the notebook is opened from `fraud-service/ml/` this matches the script exactly.


In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Embedding
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils import class_weight

# Same paths as train_lstm.py (resolved relative to the notebook's CWD)
NB_DIR = os.getcwd()
DATA_PATH  = os.path.join(NB_DIR, 'behavioral_dataset.csv')
MODEL_PATH = os.path.join(NB_DIR, 'behavior_model.h5')

print(f"TensorFlow version : {tf.__version__}")
print(f"GPU available      : {bool(tf.config.list_physical_devices('GPU'))}")
print(f"Data path          : {DATA_PATH}")
print(f"Model save path    : {MODEL_PATH}")


---

## 2. Load Data

The dataset is `behavioral_dataset.csv` - 2,000 synthetic user sessions. Each row is one session encoded as **10 sequential event codes** (`e1` ... `e10`) plus a binary `label` (`1` = fraudulent session, `0` = normal).

A few rows and a quick class-balance check follow.


In [ ]:
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"behavioral_dataset.csv not found at {DATA_PATH}. "
        "Run this notebook from fraud-service/ml/ or regenerate via generate_master_data.py."
    )

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()


In [ ]:
df.describe()


In [ ]:
# Class balance
counts = df['label'].value_counts()
ratio  = df['label'].value_counts(normalize=True).round(4)
print("Class counts:")
print(counts)
print()
print("Class ratios:")
print(ratio)


---

## 3. Visualise Behavioural Sequences

Two views - purely diagnostic, **does not affect training**:

1. **Mean event code per timestep (all sessions)** - sanity check on the synthetic data.
2. **Mean event code per timestep, fraud vs. legit** - if fraudulent sessions are mechanically uniform (bot-like), the fraud line will be flatter than the legit line.


In [ ]:
import matplotlib.pyplot as plt

event_cols = [f'e{i}' for i in range(1, 11)]

fig, ax = plt.subplots(figsize=(10, 4))
df[event_cols].mean().plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Mean event code per timestep (all sessions)')
ax.set_ylabel('Mean code')
ax.set_xlabel('Timestep')
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
mean_legit = df[df['label'] == 0][event_cols].mean()
mean_fraud = df[df['label'] == 1][event_cols].mean()

ax.plot(range(1, 11), mean_legit, marker='o', color='steelblue', label='Legit (label=0)')
ax.plot(range(1, 11), mean_fraud, marker='s', color='tomato',    label='Fraud (label=1)')
ax.set_xticks(range(1, 11))
ax.set_xlabel('Timestep (e1 ... e10)')
ax.set_ylabel('Mean event code')
ax.set_title('Behavioural sequence: legit vs. fraud')
ax.legend()
plt.tight_layout()
plt.show()


---

## 4. Preprocess

Three small steps, identical to the script:

1. **Split features and label** -> `X` (shape `(N, 10)`) and `y` (shape `(N,)`).
2. **Reshape to a 3-D tensor** `(N, 10, 1)`. Keras `LSTM` expects `(batch, timesteps, features_per_timestep)`; here we have one feature (the event code) per timestep.
3. **Compute balanced class weights** with `sklearn.utils.class_weight.compute_class_weight('balanced', ...)`. The synthetic dataset is roughly 50/50, but using class weights makes the model robust to imbalance shifts in regenerated data - and matches the script.

> Note: SMOTE is *not* used here. SMOTE interpolates in feature space, which is meaningless for ordered event codes. Class weighting is the right tool for a sequence model.


In [ ]:
X = df.drop('label', axis=1).values
y = df['label'].values

# 3-D tensor for LSTM: (samples, timesteps=10, features=1)
X = X.reshape(X.shape[0], X.shape[1], 1)

print(f"X shape : {X.shape}   (samples, timesteps, features)")
print(f"y shape : {y.shape}")


In [ ]:
weights = class_weight.compute_class_weight('balanced', classes=np.unique(y), y=y)
class_weights = dict(enumerate(weights))
print("Balanced class weights:", class_weights)


---

## 5. Build Model

Exact `Sequential` block from the script:

| Layer | Why |
| --- | --- |
| `LSTM(64, input_shape=(10, 1), return_sequences=False)` | Reads the 10-step sequence and emits a single 64-dim summary vector. `return_sequences=False` because the next layer is dense - we only need the final hidden state. |
| `Dropout(0.3)` | Stochastic regularisation; prevents the LSTM from memorising the (small) synthetic dataset. |
| `Dense(32, activation='relu')` | Non-linear projection of the LSTM summary. |
| `Dense(1, activation='sigmoid')` | Binary classification head -> fraud probability in (0, 1). |


In [ ]:
model = Sequential([
    LSTM(64, input_shape=(10, 1), return_sequences=False),
    Dropout(0.3),  # Increased dropout to prevent overfitting
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])


In [ ]:
model.summary()


---

## 6. Compile

Exact `model.compile(...)` from the script:

- **Optimiser:** Adam (default learning rate `1e-3`).
- **Loss:** `binary_crossentropy` for a two-class probabilistic head.
- **Metrics:** `accuracy`, `Precision`, `AUC`.

> **Why Precision and AUC, not just accuracy?** In TrackEasy a *false positive* triggers an SMS-OTP challenge or an outright block - a real cost to the legitimate user. We monitor Precision (FP rate) and AUC (ranking quality) explicitly so we don't deploy a high-recall model that floods the OTP tier with false alarms.


In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.AUC(name='auc'),
    ],
)


---

## 7. Train

Exact `EarlyStopping` + `model.fit(...)` from the script:

- `epochs=50`, `batch_size=32`, `validation_split=0.2`
- `class_weight=class_weights` (computed above)
- `EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)` - stops as soon as val_loss stops improving for 3 consecutive epochs and rolls back to the best weights.

The returned `history` object is captured here so the next cell can plot the training curves.


In [ ]:
print("Training with Precision monitoring and EarlyStopping...")
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(
    X, y,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    class_weight=class_weights,
    callbacks=[early_stop],
    verbose=1,
)


---

## 8. Training Curves

Visualise the four metrics across epochs (training vs. validation). A widening gap between train and val curves indicates overfitting; a flat val curve at the end indicates EarlyStopping did its job.


In [ ]:
hist = history.history

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
metrics_to_plot = [
    ('loss',      'Loss (binary cross-entropy)'),
    ('accuracy',  'Accuracy'),
    ('precision', 'Precision'),
    ('auc',       'AUC'),
]

for ax, (key, title) in zip(axes.flat, metrics_to_plot):
    if key in hist:
        ax.plot(hist[key],          label='train', color='steelblue', lw=2)
    val_key = f'val_{key}'
    if val_key in hist:
        ax.plot(hist[val_key],      label='val',   color='tomato',    lw=2)
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('LSTM training curves', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


---

## 9. Hold-out Evaluation

`validation_split=0.2` in Keras takes the **last 20% of `X` / `y`** as the validation set (no shuffling). The synthetic `behavioral_dataset.csv` is *sorted by label* (legits first, frauds last), so the Keras-internal validation slice often contains only one class - useful for the LSTM's val_loss bookkeeping but not a fair classification benchmark.

To get a meaningful classification report and confusion matrix we therefore use a **stratified 80/20 split** here. This does **not** affect training (the model has already been fit above with the original `validation_split=0.2`); it only changes the rows we *evaluate* on.

We also pass `labels=[0, 1]` so the report renders both classes even if one is empty after splitting.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Stratified split for a fair eval (training above already happened)
_, X_eval, _, y_eval = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Probabilistic predictions -> binary at the default 0.5 threshold
y_proba = model.predict(X_eval, verbose=0).ravel()
y_pred  = (y_proba >= 0.5).astype(int)

print("Eval slice class counts:", dict(zip(*np.unique(y_eval, return_counts=True))))
print()
print("Classification report (stratified 20% eval slice):")
print()
print(classification_report(
    y_eval, y_pred,
    labels=[0, 1],
    target_names=['Legit', 'Fraud'],
    zero_division=0,
))


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_eval, y_pred,
    labels=[0, 1],
    display_labels=['Legit', 'Fraud'],
    cmap='Blues',
    ax=ax,
)
ax.set_title('LSTM confusion matrix - stratified eval slice')
plt.tight_layout()
plt.show()


---

## 10. Save Model

Save to the **same path the script writes to** (`behavior_model.h5`, same folder). The `predict_*.py` services in `fraud-service/ml/` load this exact file at request time, so re-running this notebook is a drop-in replacement for `python train_lstm.py`.


In [ ]:
model.save(MODEL_PATH)
size_mb = os.path.getsize(MODEL_PATH) / (1024 * 1024)
print(f"Model saved to {MODEL_PATH}")
print(f"File size      : {size_mb:.2f} MB")


---

## 11. Next Steps

This notebook produces `behavior_model.h5`. Three sibling specialists complete *Layer 1*:

- **GNN** - [`train_gnn.py`](train_gnn.py): fraud-ring detection over shared IP / address / phone edges -> `gnn_model.h5`
- **Autoencoder** - [`train_autoencoder.py`](train_autoencoder.py): unsupervised reconstruction loss -> `autoencoder_model.h5`
- **(Master Brain)** - [`train_ann.py`](train_ann.py): consumes `lstmProb`, `gnnProb`, `autoMSE`, `ruleScore`, `geoSpeed`, `clusterSize` -> `ann_fraud_brain.h5`

Once all four `.h5` artefacts plus the joblib classical models are present, the FastAPI service in [`ml_service.py`](ml_service.py) serves the full ensemble at **http://localhost:8000/docs** (port 8000 in `docker-compose.yml`).

> **Future work:** add G-Mean alongside Precision / AUC - see TODO 1 in [`improvements_over_paper.md`](../../../../improvements_over_paper.md).
